# Data Cleaning & Structural Validation
**Dataset:** `customer_churn_raw_v2.csv` (customer churn)

**Objective:** remove duplicates, handle missing values, fix outliers and validate data types. Every decision is documented.

In [ ]:
import pandas as pd
import numpy as np

raw = pd.read_csv('customer_churn_raw_v2.csv')
df = raw.copy()
log = []   # cleaning decision log
print(raw.shape)
raw.head()

## 1. Inspect the raw data

In [ ]:
raw.info()
print('\nMissing values:\n', raw.isnull().sum()[lambda s: s > 0])
print('\nFully duplicated rows:', raw.duplicated().sum())
print('Duplicate CustomerIDs:', raw.CustomerID.duplicated().sum())
print('\nTotalChargesINR non-numeric values:')
print(raw.TotalChargesINR[pd.to_numeric(raw.TotalChargesINR, errors='coerce').isna()].value_counts())
raw.describe().T

**Issues found**
- 5 exact duplicate rows (605 rows but only 600 unique CustomerIDs)
- `Region` has 13 missing values
- `TotalChargesINR` is stored as text because 11 rows contain the string `unknown`
- `MonthlyChargesINR` has 6 extreme outliers (about 12.5x too high for their plan) plus 1 more in a row with unknown total
- `TenureMonths = 0` for some customers (valid: brand-new customers)

## 2. Remove duplicates
The duplicate rows are identical in every column, so keeping the first occurrence loses no information.

In [ ]:
before = len(df)
df = df.drop_duplicates(keep='first').reset_index(drop=True)
log.append(('Duplicates', f'Dropped {before-len(df)} exact duplicate rows', 'Identical in all columns; keeping first loses nothing'))
assert df.CustomerID.is_unique
print(before, '->', len(df))

## 3. Standardize headers and text values

In [ ]:
import re
def to_snake(c):
    c = re.sub(r'(?<=[a-z])(?=[A-Z])', '_', c)
    return c.lower()
df.columns = [to_snake(c) for c in df.columns]
log.append(('Headers', 'Converted column names to snake_case', 'Consistent, easy to use in code'))

text_cols = df.select_dtypes(include=['object', 'string']).columns
for c in text_cols:
    df[c] = df[c].str.strip()
for c in ['subscription_type', 'contract_type', 'payment_method', 'region']:
    df[c] = df[c].str.title()
df['payment_method'] = df['payment_method'].replace({'Upi': 'UPI'})
log.append(('Categoricals', 'Trimmed whitespace and applied consistent casing', 'Prevents the same category appearing under different spellings'))
df.columns.tolist()

## 4. Fix data types and the `unknown` placeholder
`total_charges_inr` must be numeric. `unknown` is a placeholder for a missing value, so it is converted to `NaN` first.

In [ ]:
df['total_charges_inr'] = pd.to_numeric(df['total_charges_inr'].replace('unknown', np.nan), errors='coerce')
log.append(('Data types', "Converted total_charges_inr to float; 'unknown' -> NaN", 'Text placeholder blocked numeric analysis'))

for c in ['auto_pay', 'churn']:
    df[c] = df[c].map({'Yes': 1, 'No': 0}).astype('int8')
log.append(('Data types', 'Encoded auto_pay and churn as 1/0', 'Binary flags are easier for analysis and modelling'))
df.dtypes

## 5. Handle outliers in `monthly_charges_inr`
For each subscription type the 95th percentile is within about 6% of the median, so prices are tightly clustered. Seven values are roughly 10-30x higher, and in six of them `total / (monthly x tenure)` is about 0.08 instead of about 1.0, so the monthly value is clearly inflated (a data-entry or scaling error). Those values are replaced with the median monthly charge of the same plan (median is robust to the outliers themselves).

In [ ]:
grp = df.groupby('subscription_type')['monthly_charges_inr']
med = grp.transform('median')
q3 = grp.transform(lambda s: s.quantile(0.75))
iqr = grp.transform(lambda s: s.quantile(0.75) - s.quantile(0.25))
out = df['monthly_charges_inr'] > (q3 + 3 * iqr)
print('Outliers flagged:', out.sum())
display(df.loc[out, ['customer_id', 'subscription_type', 'tenure_months', 'monthly_charges_inr', 'total_charges_inr']])

df.loc[out, 'monthly_charges_inr'] = med[out].round(2)
log.append(('Outliers', f'Replaced {out.sum()} inflated monthly_charges_inr values with the median of their subscription type', 'About 12.5x too high vs plan price and vs total charges / tenure; median is robust'))
df.groupby('subscription_type')['monthly_charges_inr'].describe()

## 6. Handle missing values
**`total_charges_inr`** (11 missing): total charges should equal monthly charge x tenure. For existing customers with valid totals the ratio is about 0.98 (median), so the estimate is reliable. Customers with 0 months tenure get 0. This is better than the mean or dropping rows (dropping would lose valid customers).

**`region`** (missing): it is a nominal category, so no value can be inferred honestly. It is filled with `Unknown` instead of guessing, which also avoids biasing regional analysis.

In [ ]:
miss = df['total_charges_inr'].isna()
df.loc[miss, 'total_charges_inr'] = (df.loc[miss, 'monthly_charges_inr'] * df.loc[miss, 'tenure_months']).round(2)
log.append(('Missing values', f'Imputed {miss.sum()} total_charges_inr as monthly_charges x tenure', 'Total charges is cumulative billing; ratio to monthly x tenure is about 0.98 in valid rows'))

r_missing = df['region'].isna().sum()
df['region'] = df['region'].fillna('Unknown')
log.append(('Missing values', f'Filled {r_missing} missing region values with "Unknown"', 'Nominal category cannot be inferred; avoids fabricating data'))
print(df.isnull().sum().sum(), 'missing values left')

## 7. Validation checks

In [ ]:
checks = {
 'No missing values': df.isnull().sum().sum() == 0,
 'No duplicate rows': df.duplicated().sum() == 0,
 'customer_id unique': df.customer_id.is_unique,
 'age in 18-100': df.age.between(18, 100).all(),
 'tenure 0-72': df.tenure_months.between(0, 72).all(),
 'charges non-negative': (df[['monthly_charges_inr', 'total_charges_inr']] >= 0).all().all(),
 'total approx monthly x tenure (+-25%)': (df.total_charges_inr.sub(df.monthly_charges_inr * df.tenure_months).abs() <= 0.25 * (df.monthly_charges_inr * df.tenure_months) + 1).all(),
 'numeric dtypes': pd.api.types.is_float_dtype(df.total_charges_inr),
}
for k, v in checks.items():
    print('PASS' if v else 'FAIL', '-', k)
assert all(checks.values())

## 8. Export

In [ ]:
df.to_csv('customer_churn_cleaned.csv', index=False)
log_df = pd.DataFrame(log, columns=['Area', 'Action', 'Reasoning'])
log_df.to_csv('cleaning_log.csv', index=False)
print(f'Raw: {raw.shape}  ->  Cleaned: {df.shape}')
log_df

## Summary
| Issue | Count | Action |
|---|---|---|
| Duplicate rows | 5 | Dropped (kept first) |
| `unknown` in total charges | 10 after dedup | Converted to NaN, imputed monthly x tenure |
| Missing region | see log | Filled with `Unknown` |
| Monthly charge outliers | 7 | Replaced with plan median |
| Text columns / headers | all | Trimmed, consistent case, snake_case |
| Data types | 3 | total charges to float, yes/no flags to 1/0 |

Final dataset: 600 rows x 13 columns, with no missing values or duplicates.